# Silent Middle - Comparison of Approaches

This notebook compares two approaches:

1. **Original Approach**: Study-relative definition (conformist) + single CV
2. **Improved Approach**: 
   - Alternative definition: Absolute middle of scale (moderate)
   - Nested CV for proper hyperparameter validation

Both approaches compared side-by-side.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedGroupKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)

## 1. Load and Prepare Data

In [2]:
# Load data
raw_df = pd.read_excel('Data files/fulldataset.xlsx')
print(f"Dataset shape: {raw_df.shape}")
print(f"Number of studies: {raw_df['study'].nunique()}")

Dataset shape: (56329, 91)
Number of studies: 20


In [3]:
# Identify choice columns
ct1_cols = [c for c in raw_df.columns if c.startswith("ct1_")]
ct2_cols = [c for c in raw_df.columns if c.startswith("ct2_")]
ct_cols = ct1_cols + ct2_cols
mct_cols = [c for c in raw_df.columns if c.startswith("motct")]

print(f"Choice columns: {len(ct_cols)}")
print(f"Motivation columns: {len(mct_cols)}")

# Ensure numeric
df = raw_df.copy()
for c in ct_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

Choice columns: 30
Motivation columns: 30


## 2. Create Target Variables - Two Approaches

### Approach 1: Study-Relative (Original - "Conformist")
- Measures deviation from study-specific median
- "Silent middle" = people with typical/average opinions for that study

### Approach 2: Absolute Middle (Alternative - "Moderate")
- Measures deviation from scale midpoint (0.5 after normalization)
- "Silent middle" = people with moderate/neutral opinions regardless of others

In [4]:
def create_study_relative_target(df, ct_cols, study_col="study"):
    """
    Original approach: Deviation from study median
    """
    def add_deviation_scores(group):
        # Study-specific medians
        med = group[ct_cols].median(numeric_only=True)
        
        # Deviation from study median
        deviation = group[ct_cols].sub(med, axis=1)
        abs_deviation = deviation.abs()
        
        # Mean absolute deviation per person
        mad = abs_deviation.mean(axis=1, skipna=True)
        
        group = group.copy()
        group["ct_mad_from_median"] = mad
        return group
    
    df_temp = df.groupby(study_col, group_keys=False).apply(add_deviation_scores)
    
    # Threshold at study median
    study_median_mad = df_temp.groupby(study_col)["ct_mad_from_median"].transform("median")
    df_temp["silent_middle_relative"] = (df_temp["ct_mad_from_median"] <= study_median_mad).astype(int)
    
    return df_temp


def create_absolute_middle_target(df, ct_cols, study_col="study"):
    """
    Alternative approach: Deviation from scale midpoint
    For each option in each study, normalize to [0,1] then measure deviation from 0.5
    """
    def add_absolute_deviation(group):
        # For each choice column, get min and max in THIS study
        normalized = pd.DataFrame(index=group.index)
        
        for col in ct_cols:
            col_min = group[col].min()
            col_max = group[col].max()
            
            # Normalize to [0, 1]
            if col_max > col_min:
                normalized[col] = (group[col] - col_min) / (col_max - col_min)
            else:
                # All same value - treat as middle
                normalized[col] = 0.5
        
        # Deviation from absolute middle (0.5)
        abs_deviation = (normalized - 0.5).abs()
        
        # Mean absolute deviation from midpoint
        mad_absolute = abs_deviation.mean(axis=1, skipna=True)
        
        group = group.copy()
        group["ct_mad_from_midpoint"] = mad_absolute
        return group
    
    df_temp = df.groupby(study_col, group_keys=False).apply(add_absolute_deviation)
    
    # Threshold at study median (to get ~50%)
    study_median_mad_abs = df_temp.groupby(study_col)["ct_mad_from_midpoint"].transform("median")
    df_temp["silent_middle_absolute"] = (df_temp["ct_mad_from_midpoint"] <= study_median_mad_abs).astype(int)
    
    return df_temp

In [5]:
# Create both target variables
print("Creating study-relative target (original approach)...")
df = create_study_relative_target(df, ct_cols)

print("Creating absolute middle target (alternative approach)...")
df = create_absolute_middle_target(df, ct_cols)

print("\nTarget variable distributions:")
print(f"Study-relative (conformist): {df['silent_middle_relative'].mean():.1%} silent")
print(f"Absolute middle (moderate):  {df['silent_middle_absolute'].mean():.1%} silent")
print(f"\nOverlap: {(df['silent_middle_relative'] == df['silent_middle_absolute']).mean():.1%} same classification")

Creating study-relative target (original approach)...
Creating absolute middle target (alternative approach)...

Target variable distributions:
Study-relative (conformist): 53.5% silent
Absolute middle (moderate):  66.9% silent

Overlap: 59.2% same classification


## 3. Feature Engineering

In [6]:
# Motivation features
df["motivationamount"] = df[mct_cols].notna().sum(axis=1)
df["motivation_length"] = (
    df[mct_cols]
    .fillna("")
    .astype(str)
    .apply(lambda row: sum(len(s.split()) for s in row), axis=1)
)
df["logmotivationlength"] = np.log1p(df["motivation_length"])

print(f"Motivation features created")
print(f"  Amount range: {df['motivationamount'].min():.0f} - {df['motivationamount'].max():.0f}")
print(f"  Length range: {df['motivation_length'].min():.0f} - {df['motivation_length'].max():.0f} words")

Motivation features created
  Amount range: 0 - 22
  Length range: 0 - 3656 words


In [7]:
# Define features to exclude (prevent data leakage)
leakage_cols = set(ct_cols + ["ct_mad_from_median", "ct_mad_from_midpoint", 
                               "silent_middle_relative", "silent_middle_absolute"])
motivation_text_cols = set(mct_cols)
study_structure_cols = {"study", "ct1type", "ct2type"}
id_like = {"id", "time"}
exclude = leakage_cols | motivation_text_cols | study_structure_cols | id_like | {"motivation_length"}

# Build feature matrix
X = df[[c for c in df.columns if c not in exclude]].copy()
groups = df["study"].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Features: {list(X.columns)}")

Feature matrix shape: (56329, 28)
Features: ['panel', 'level', 'sector', 'lossesgains', 'grade', 'age', 'education', 'gender', 'finance', 'dailylife', 'health', 'home', 'nevergiaveadvice', 'giveadvicesurvey', 'giveadvicemeeting', 'giveadviceinterestgroup', 'giveadviceother', 'advice', 'choicesteering', 'honestresearch', 'subjectimportant', 'hardunderstanding', 'usemoreoften', 'choiceslearning', 'decisionaccepting', 'decisiontrusting', 'motivationamount', 'logmotivationlength']


## 4. Setup Preprocessing Pipeline

In [8]:
def create_preprocessing_pipeline(X):
    """
    Creates a preprocessing pipeline for the feature matrix
    """
    cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns
    num_cols = X.columns.difference(cat_cols)
    
    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median"))
            ]), num_cols),
            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("oh", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
        ]
    )
    
    return preprocess

preprocess = create_preprocessing_pipeline(X)
print("Preprocessing pipeline created")

Preprocessing pipeline created


## 5. Model Training - Two Approaches

### Approach 1: Single CV (Original)
- Use same CV splits for hyperparameter tuning and evaluation
- Faster but slight optimistic bias

### Approach 2: Nested CV (Improved)
- Outer loop: Evaluation
- Inner loop: Hyperparameter tuning
- More conservative, unbiased estimates

In [9]:
# Hyperparameter space (same for both)
hyperparameter_space = {
    "rf__n_estimators": [150, 250, 350],
    "rf__max_depth": [4, 6, 8],
    "rf__min_samples_leaf": [30, 50, 80],
    "rf__min_samples_split": [30, 50],
    "rf__max_features": [0.1, 0.2, 0.3],
    "rf__bootstrap": [True],
}

n_splits_outer = 5
n_splits_inner = 3  # For nested CV inner loop
n_iter = 30  # Reduced for faster execution

In [10]:
def train_single_cv(X, y, groups, preprocess, hyperparameter_space, n_splits=5, n_iter=30):
    """
    Original approach: Single CV for both tuning and evaluation
    """
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
    rf_pipe = Pipeline(steps=[("prep", preprocess), ("rf", rf)])
    
    rf_search = RandomizedSearchCV(
        rf_pipe,
        param_distributions=hyperparameter_space,
        n_iter=n_iter,
        scoring="roc_auc",
        refit=True,
        cv=cv,
        n_jobs=1,
        random_state=42,
        verbose=0,
    )
    
    print("  Training with single CV...")
    rf_search.fit(X, y, groups=groups)
    
    # Get out-of-fold predictions using SAME cv
    best_model = rf_search.best_estimator_
    y_pred_proba = cross_val_predict(best_model, X, y, cv=cv, groups=groups, method="predict_proba", n_jobs=-1)[:, 1]
    y_pred = cross_val_predict(best_model, X, y, cv=cv, groups=groups, method="predict", n_jobs=-1)
    
    return rf_search, y_pred_proba, y_pred, best_model


def train_nested_cv(X, y, groups, preprocess, hyperparameter_space, n_splits_outer=5, n_splits_inner=3, n_iter=30):
    """
    Improved approach: Nested CV for unbiased evaluation
    """
    outer_cv = StratifiedGroupKFold(n_splits=n_splits_outer, shuffle=True, random_state=42)
    inner_cv = StratifiedGroupKFold(n_splits=n_splits_inner, shuffle=True, random_state=43)
    
    rf = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
    rf_pipe = Pipeline(steps=[("prep", preprocess), ("rf", rf)])
    
    rf_search = RandomizedSearchCV(
        rf_pipe,
        param_distributions=hyperparameter_space,
        n_iter=n_iter,
        scoring="roc_auc",
        refit=True,
        cv=inner_cv,  # Inner CV for hyperparameter tuning
        n_jobs=1,
        random_state=42,
        verbose=0,
    )
    
    print("  Training with nested CV (outer loop for evaluation)...")
    
    # Outer CV for evaluation
    y_pred_proba = np.zeros(len(y))
    y_pred = np.zeros(len(y))
    
    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups)):
        print(f"    Outer fold {fold_idx + 1}/{n_splits_outer}", end="\r")
        
        X_train_fold = X.iloc[train_idx]
        y_train_fold = y.iloc[train_idx]
        groups_train_fold = groups.iloc[train_idx]
        
        X_test_fold = X.iloc[test_idx]
        
        # Fit RandomizedSearchCV on training fold (uses inner CV)
        rf_search.fit(X_train_fold, y_train_fold, groups=groups_train_fold)
        
        # Predict on test fold
        y_pred_proba[test_idx] = rf_search.predict_proba(X_test_fold)[:, 1]
        y_pred[test_idx] = rf_search.predict(X_test_fold)
    
    print("\n  Nested CV complete")
    
    # Fit final model on all data for feature importance
    print("  Fitting final model on full dataset...")
    rf_search.fit(X, y, groups=groups)
    final_model = rf_search.best_estimator_
    
    return rf_search, y_pred_proba, y_pred, final_model

## 6. Train All Four Combinations

1. Study-relative target + Single CV (original)
2. Study-relative target + Nested CV
3. Absolute middle target + Single CV
4. Absolute middle target + Nested CV

In [11]:
results = {}

# Approach 1: Study-relative + Single CV
print("\n" + "="*60)
print("APPROACH 1: Study-Relative Target + Single CV (Original)")
print("="*60)
y_relative = df["silent_middle_relative"]
search_rel_single, proba_rel_single, pred_rel_single, model_rel_single = train_single_cv(
    X, y_relative, groups, create_preprocessing_pipeline(X), hyperparameter_space, n_splits=n_splits_outer, n_iter=n_iter
)   
results['relative_single'] = {
    'search': search_rel_single,
    'proba': proba_rel_single,
    'pred': pred_rel_single,
    'y': y_relative,
    'model': model_rel_single
}


APPROACH 1: Study-Relative Target + Single CV (Original)
  Training with single CV...


KeyboardInterrupt: 

In [ ]:
# Approach 2: Study-relative + Nested CV
print("\n" + "="*60)
print("APPROACH 2: Study-Relative Target + Nested CV (Improved CV)")
print("="*60)
search_rel_nested, proba_rel_nested, pred_rel_nested, model_rel_nested = train_nested_cv(
    X, y_relative, groups, create_preprocessing_pipeline(X), hyperparameter_space, 
    n_splits_outer=n_splits_outer, n_splits_inner=n_splits_inner, n_iter=n_iter
)
results['relative_nested'] = {
    'search': search_rel_nested,
    'proba': proba_rel_nested,
    'pred': pred_rel_nested,
    'y': y_relative,
    'model': model_rel_nested
}

In [ ]:
# Approach 3: Absolute middle + Single CV
print("\n" + "="*60)
print("APPROACH 3: Absolute Middle Target + Single CV")
print("="*60)
y_absolute = df["silent_middle_absolute"]
search_abs_single, proba_abs_single, pred_abs_single, model_abs_single = train_single_cv(
    X, y_absolute, groups, create_preprocessing_pipeline(X), hyperparameter_space, n_splits=n_splits_outer, n_iter=n_iter
)
results['absolute_single'] = {
    'search': search_abs_single,
    'proba': proba_abs_single,
    'pred': pred_abs_single,
    'y': y_absolute,
    'model': model_abs_single
}

In [ ]:
# Approach 4: Absolute middle + Nested CV
print("\n" + "="*60)
print("APPROACH 4: Absolute Middle Target + Nested CV (Best Practice)")
print("="*60)
search_abs_nested, proba_abs_nested, pred_abs_nested, model_abs_nested = train_nested_cv(
    X, y_absolute, groups, create_preprocessing_pipeline(X), hyperparameter_space,
    n_splits_outer=n_splits_outer, n_splits_inner=n_splits_inner, n_iter=n_iter
)
results['absolute_nested'] = {
    'search': search_abs_nested,
    'proba': proba_abs_nested,
    'pred': pred_abs_nested,
    'y': y_absolute,
    'model': model_abs_nested
}

## 7. Compare Results

In [ ]:
def calculate_metrics(y_true, y_pred_proba, y_pred):
    """
    Calculate comprehensive metrics
    """
    metrics = {
        'ROC AUC': roc_auc_score(y_true, y_pred_proba),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'R²': r2_score(y_true, y_pred_proba),
        'MSE': mean_squared_error(y_true, y_pred_proba),
    }
    return metrics

# Calculate metrics for all approaches
comparison = pd.DataFrame()

for name, data in results.items():
    metrics = calculate_metrics(data['y'], data['proba'], data['pred'])
    comparison[name] = pd.Series(metrics)

# Rename columns for clarity
comparison.columns = [
    'Relative + Single CV\n(Original)',
    'Relative + Nested CV',
    'Absolute + Single CV',
    'Absolute + Nested CV\n(Best Practice)'
]

print("\n" + "="*80)
print("PERFORMANCE COMPARISON")
print("="*80)
print(comparison.round(4))
print("\n" + "="*80)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Performance Comparison Across Approaches', fontsize=14, fontweight='bold')

metrics_to_plot = ['ROC AUC', 'Accuracy', 'Precision', 'Recall', 'R²', 'MSE']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 3, idx % 3]
    data = comparison.loc[metric]
    bars = ax.bar(range(len(data)), data.values, color=colors)
    ax.set_xticks(range(len(data)))
    ax.set_xticklabels(data.index, rotation=45, ha='right', fontsize=8)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 8. Key Findings

### Effect of Nested CV:
- Compare columns 1 vs 2 (Relative) and 3 vs 4 (Absolute)
- Nested CV typically shows slightly lower (more conservative) performance estimates
- Difference indicates optimistic bias in single CV approach

### Effect of Target Definition:
- Compare rows: Relative vs Absolute
- **Study-relative (conformist)**: Identifies people with typical/average opinions
- **Absolute middle (moderate)**: Identifies people with neutral/middle-of-scale opinions

### Which approach predicts better?
- Higher ROC AUC = better discrimination
- Higher R² = better probability calibration

In [ ]:
# Best hyperparameters from each approach
print("\nBest Hyperparameters:")
print("\n1. Relative + Single CV:")
print(search_rel_single.best_params_)

print("\n2. Relative + Nested CV:")
print(search_rel_nested.best_params_)

print("\n3. Absolute + Single CV:")
print(search_abs_single.best_params_)

print("\n4. Absolute + Nested CV:")
print(search_abs_nested.best_params_)

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Confusion Matrices - All Approaches', fontsize=14, fontweight='bold')

approaches = [
    ('relative_single', 'Relative + Single CV\n(Original)', axes[0, 0]),
    ('relative_nested', 'Relative + Nested CV', axes[0, 1]),
    ('absolute_single', 'Absolute + Single CV', axes[1, 0]),
    ('absolute_nested', 'Absolute + Nested CV\n(Best Practice)', axes[1, 1]),
]

for key, title, ax in approaches:
    data = results[key]
    cm = confusion_matrix(data['y'], data['pred'], labels=[1, 0])
    
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Silent', 'Not Silent']
    )
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(title, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Interpretation

### Single CV vs Nested CV:
- **Single CV**: Faster, slightly optimistic (same folds for tuning & evaluation)
- **Nested CV**: More rigorous, unbiased estimates (separate folds)
- **Difference**: Usually small if model is well-regularized

### Study-Relative vs Absolute Middle:

**Study-Relative (Conformist):**
- Pro: Accounts for topic-specific norms
- Pro: Works across different scales
- Con: "Silent" can mean strongly opinionated if everyone agrees

**Absolute Middle (Moderate):**
- Pro: Identifies truly neutral/undecided respondents
- Pro: Intuitive interpretation
- Con: Sensitive to scale differences across choice tasks

### Recommendation:
For the assignment, **study-relative + nested CV** is likely best:
- Scientifically rigorous (nested CV)
- Appropriate for varying scales (study-relative)
- Aligns with research question (understanding participation patterns)